# Klasifikasi Kelayakan Kapal Penangkap Ikan Menggunakan Histogram of Oriented Gradients (HOG) dan Support Vector Machine (SVM)

Notebook ini disusun sebagai referensi penelitian untuk melakukan pelatihan model, pengujian, serta evaluasi klasifikasi kelayakan fisik lambung kapal (kategori **Baik** dan **Rusak**). Penjelasan teoretis serta rumus matematis disertakan pada setiap tahap untuk menunjang penyusunan bab analisis skripsi atau laporan ilmiah.

---


## 1. Import Library

Modul yang dipanggil untuk penelitian:
1. `numpy`, `pandas`: Digunakan untuk manipulasi aljabar linear dan struktur data tabel.
2. `skimage.feature.hog`: Implementasi ekstraksi fitur tekstur dan bentuk (Dalal & Triggs, 2005).
3. `sklearn.svm.SVC`: Implementasi klasifikasi margin maksimal dan trik kernel (Vapnik, 1995).
4. `matplotlib`, `seaborn`: Visualisasi performa model (Heatmap, Histogram).
5. `cv2` (OpenCV): Pemrosesan gambar matriks.


In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from skimage.feature import hog
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# Set seed for reproducibility and consistency
np.random.seed(42)


## 2. Load Dataset

Dataset terdiri atas dua kelas:
- **Kelas Positif (1)**: Kapal dalam kondisi fisik **Baik**.
- **Kelas Negatif (0)**: Kapal dalam kondisi fisik **Rusak** (korosi, retak, kebocoran).


In [ ]:
DATASET_PATH = 'dataset'
CATEGORIES = ['baik', 'rusak']

def load_data(dataset_path, categories):
    image_paths = []
    labels = []
    
    for category in categories:
        folder_path = os.path.join(dataset_path, category)
        label = 1 if category == 'baik' else 0 # 1 untuk Baik, 0 untuk Rusak
        
        if not os.path.exists(folder_path):
            print(f"Warning: Folder {folder_path} tidak ditemukan.")
            continue
            
        for filename in os.listdir(folder_path):
            if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                img_path = os.path.join(folder_path, filename)
                image_paths.append(img_path)
                labels.append(label)
                
    return image_paths, labels

image_paths, labels = load_data(DATASET_PATH, CATEGORIES)
print(f"Total gambar ditemukan: {len(image_paths)}")
print(f"Total label: {len(labels)}")


## 3. Preprocessing (Pra-pemrosesan Citra)

Sebelum fitur diekstraksi, gambar mentah $I(x,y)$ dikonversi menjadi citra *grayscale* agar perhitungan gradien berfokus pada intensitas cahaya (pola bentuk) bukan intensitas warna (RGB).

Citra di-*resize* ke dimensi $128 \times 128$ untuk menormalisasi ukuran vektor keluaran.


In [ ]:
IMG_SIZE = (128, 128)

def preprocess_image(img_path):
    img = cv2.imread(img_path)
    if img is None:
        raise ValueError(f"Gagal membaca gambar: {img_path}")
        
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, IMG_SIZE)
    img_gray = cv2.cvtColor(img_resized, cv2.COLOR_RGB2GRAY)
    
    return img_rgb, img_gray

if len(image_paths) > 0:
    sample_path = image_paths[0]
    img_rgb, img_gray = preprocess_image(sample_path)
    
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(img_rgb)
    axes[0].set_title("Original RGB (Resized)")
    axes[0].axis('off')
    axes[1].imshow(img_gray, cmap='gray')
    axes[1].set_title("Grayscale")
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()


## 4. Ekstraksi Fitur HOG (Histogram of Oriented Gradients)

Metode HOG mengekstrak struktur lokal benda dengan menghitung distribusi arah piksel lokal (gradien intensitas).

1. **Magnitudo dan Orientasi Gradien**
   Pada setiap piksel $(x,y)$, komponen gradien horizontal ($G_x$) dan vertikal ($G_y$) dihitung menggunakan filter derivatif sederhana (misal $[-1, 0, 1]$):
   $$ G_x(x,y) = I(x+1, y) - I(x-1, y) $$
   $$ G_y(x,y) = I(x, y+1) - I(x, y-1) $$
   Magnitudo ($m$) dan Sudut / Orientasi ($\theta$) dihitung dengan persamaan:
   $$ m(x,y) = \sqrt{G_x(x,y)^2 + G_y(x,y)^2} $$
   $$ \theta(x,y) = \arctan\left(\frac{G_y(x,y)}{G_x(x,y)}\right) $$

2. **Histogram dalam Cell**
   Piksel dibagi ke dalam beberapa sel (contoh: $8 \times 8$ piksel). Setiap piksel menyumbangkan magnitudonya ke bin (rentang orientasi, contoh: 9 orientasi $0^\circ - 180^\circ$).

3. **Block Normalization (L2-Hys)**
   Untuk menolerir variasi pencahayaan, sel dikelompokkan menjadi blok (contoh: $2 \times 2$ sel), lalu dilakukan normalisasi blok dengan persamaan *L2-norm*:
   $$ v = \frac{v}{\sqrt{\|v\|^2 + \epsilon^2}} $$


In [ ]:
HOG_ORIENTATIONS = 9
HOG_PIXELS_PER_CELL = (8, 8)
HOG_CELLS_PER_BLOCK = (2, 2)
HOG_BLOCK_NORM = 'L2-Hys'

def extract_hog_features(img_gray, visualize=False):
    if visualize:
        features, hog_image = hog(
            img_gray,
            orientations=HOG_ORIENTATIONS,
            pixels_per_cell=HOG_PIXELS_PER_CELL,
            cells_per_block=HOG_CELLS_PER_BLOCK,
            block_norm=HOG_BLOCK_NORM,
            visualize=True,
            transform_sqrt=True
        )
        return features, hog_image
    else:
        features = hog(
            img_gray,
            orientations=HOG_ORIENTATIONS,
            pixels_per_cell=HOG_PIXELS_PER_CELL,
            cells_per_block=HOG_CELLS_PER_BLOCK,
            block_norm=HOG_BLOCK_NORM,
            visualize=False,
            transform_sqrt=True
        )
        return features

if len(image_paths) > 0:
    features, hog_image = extract_hog_features(img_gray, visualize=True)
    
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(img_gray, cmap='gray')
    axes[0].set_title("Grayscale Image")
    axes[0].axis('off')
    
    axes[1].imshow(hog_image, cmap='gray')
    axes[1].set_title("HOG Visualization")
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
    print(f"Dimensi fitur HOG keluaran: {features.shape}")


## 5. Persiapan Dataset (Pembuatan DataFrame)

Proses perulangan di mana setiap citra dikonversi menjadi deretan fitur array satu dimensi. Kumpulan vektor fitur tersebut dirakit menjadi sebuah matriks data $\mathbf{X}$ (variabel independen) dengan label $\mathbf{y}$ (variabel dependen).


In [ ]:
X = []
y = []

print("Mengekstraksi fitur HOG untuk seluruh data...")
for path, label in zip(image_paths, labels):
    try:
        _, img_gray = preprocess_image(path)
        features = extract_hog_features(img_gray)
        X.append(features)
        y.append(label)
    except Exception as e:
        pass

X = np.array(X)
y = np.array(y)

if len(X) > 0:
    feature_cols = [f'hog_{i}' for i in range(X.shape[1])]
    df = pd.DataFrame(X, columns=feature_cols)
    df['label'] = y
    
    print(f"Total data diproses: {len(df)}")
    display(df.head())


## 6. Train-Test Split (Pembagian Data)

Membagi data Latih (Training) dan data Uji (Testing) sangat krusial dalam klasifikasi ML untuk memvalidasi seberapa baik kemampuan abstraksi/generalization model terhadap data baru.


In [ ]:
if len(X) > 0:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, 
        test_size=0.20, 
        random_state=42, 
        stratify=y
    )
    print(f"Dimensi data training: {X_train.shape}")
    print(f"Dimensi data testing: {X_test.shape}")


## 7. Feature Scaling (Standardisasi Z-Score)

Berbeda dengan Tree-based model, SVM sangat sensitif terhadap skala fitur dikarenakan perhitungannya bergantung pada jarak Euclidean antar vektor. Z-Score Normalization diterapkan untuk mendistribusikan setiap fitur memiliki rata-rata (mean) = 0 dan variansi = 1.

$$ z = \frac{x - \mu}{\sigma} $$

Standar acuan $\mu$ dan $\sigma$ dari data *Training* disimpan ke `scaler.pkl` agar kelak fitur baru (*Testing* atau pada *Deploy Flask*) dapat dikenakan standarisasi dengan batas proporsi yang sama persis.


In [ ]:
if len(X) > 0:
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    os.makedirs('models', exist_ok=True)
    scaler_path = 'models/scaler.pkl'
    joblib.dump(scaler, scaler_path)
    print(f"Scaler berhasil disimpan di: {scaler_path}")


## 8. Pemodelan Support Vector Machine (SVM)

Ide dasar klasifikasi SVM adalah mencari bidang batas (Hyperplane) yang memiliki margin tersebar secara geometri di antara himpunan data dua kelas yang saling bertentangan (Baik / Rusak).

Persamaan batas (decision function) dari SVM:
$$ f(x) = \text{sign} \left( \sum_{i=1}^{n} \alpha_i y_i K(x_i, x) + b \right) $$

Dimana:
- $\alpha_i$: Pengali/Bobot Lagrange (bobot penalti). Titik fitur yang tidak memiliki nilai nol pada elemen ini disebut *Support Vectors*.
- $K(x_i, x)$: Fungsi Kernel. Digunakan agar data yang kompleks dapat secara virtual diproyeksikan ke dimensi yang jauh lebih tinggi dan dapat dipisahkan secara linier (Kernel Trick).

### Evaluasi Kombinasi Parameter (Grid Search)
Dalam penelitian ini, *tuning hyperparameter* diaplikasikan pada:
1. **Kernel Linear**: Menggunakan garis pisah linier langsung.
2. **Kernel RBF (Radial Basis Function)**: Menggunakan kurva bel (Gaussian), memiliki formula matematis $K(x, x') = \exp(-\gamma \|x - x'\|^2)$.
- Parameter $C$: Biaya pinalti terhadap misklasifikasi (mengontrol margin *soft*/*hard*).
- Parameter $\gamma$: Variabel penyebaran pengaruh support vector di ruang fitur untuk RBF.


In [ ]:
if len(X) > 0:
    print("Melatih SVM Linear...")
    param_grid_linear = {'C': [0.1, 1, 10, 100], 'kernel': ['linear']}
    grid_linear = GridSearchCV(SVC(probability=True, random_state=42), param_grid_linear, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
    grid_linear.fit(X_train_scaled, y_train)
    best_linear_model = grid_linear.best_estimator_
    print(f"Best parameter Linear: {grid_linear.best_params_}")
    
    print("\nMelatih SVM RBF...")
    param_grid_rbf = {'C': [0.1, 1, 10, 100], 'gamma': [0.001, 0.01, 0.1, 1], 'kernel': ['rbf']}
    grid_rbf = GridSearchCV(SVC(probability=True, random_state=42), param_grid_rbf, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
    grid_rbf.fit(X_train_scaled, y_train)
    best_rbf_model = grid_rbf.best_estimator_
    print(f"Best parameter RBF: {grid_rbf.best_params_}")


## 9. Evaluasi Performa (Confusion Matrix Metrics)

Berdasarkan *Confusion Matrix* (Matriks Kesalahan):
- **TP (True Positive)**: Prediksi "Baik", Asli "Baik".
- **TN (True Negative)**: Prediksi "Rusak", Asli "Rusak".
- **FP (False Positive)**: Prediksi "Baik", Asli "Rusak" (Salah Deteksi).
- **FN (False Negative)**: Prediksi "Rusak", Asli "Baik" (Lewat Deteksi).

Penelitian ini menggunakan 4 perhitungan performa dasar:
1. **Accuracy**: Total prediksi benar dari seluruh populasi prediksi.
   $$ \text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN} $$
2. **Precision**: Kualitas presisi prediksi positif (Akurasi deteksi "Baik").
   $$ \text{Precision} = \frac{TP}{TP + FP} $$
3. **Recall**: Sensitivitas menangkap *True Positives* asli.
   $$ \text{Recall} = \frac{TP}{TP + FN} $$
4. **F1-Score**: Rata-rata harmonik penyeimbang di mana *Precision* dan *Recall* sama-sama penting.
   $$ F1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}} $$


In [ ]:
def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    print(f"--- Evaluasi {model_name} ---")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1 Score : {f1:.4f}\n")
    
    cm = confusion_matrix(y_test, y_pred)
    return acc, prec, rec, f1, cm

if len(X) > 0:
    acc_lin, prec_lin, rec_lin, f1_lin, cm_lin = evaluate_model(best_linear_model, X_test_scaled, y_test, "SVM Linear")
    acc_rbf, prec_rbf, rec_rbf, f1_rbf, cm_rbf = evaluate_model(best_rbf_model, X_test_scaled, y_test, "SVM RBF")


## 10. Visualisasi dan Penentuan Model Terbaik


In [ ]:
if len(X) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    sns.heatmap(cm_lin, annot=True, fmt='d', cmap='Blues', ax=axes[0], 
                xticklabels=['Rusak (0)', 'Baik (1)'], yticklabels=['Rusak', 'Baik'])
    axes[0].set_title('Confusion Matrix - SVM Linear')
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('Actual')
    
    sns.heatmap(cm_rbf, annot=True, fmt='d', cmap='Greens', ax=axes[1],
                xticklabels=['Rusak (0)', 'Baik (1)'], yticklabels=['Rusak', 'Baik'])
    axes[1].set_title('Confusion Matrix - SVM RBF')
    axes[1].set_xlabel('Predicted')
    axes[1].set_ylabel('Actual')
    
    plt.tight_layout()
    plt.show()
    
    best_model = best_linear_model if acc_lin >= acc_rbf else best_rbf_model
    best_model_name = "SVM Linear" if acc_lin >= acc_rbf else "SVM RBF"
    best_acc = max(acc_lin, acc_rbf)
    
    print(f"\n[KESIMPULAN] Model SVM terbaik adalah {best_model_name} dengan Accuracy {best_acc*100:.2f}%.")
    
    model_path = 'models/svm_model.pkl'
    joblib.dump(best_model, model_path)
    print(f"Model SVM berhasil diekspor menuju: {model_path} untuk integrasi ke arsitektur Web Flask.")


## 11. Simulasi Workflow Inferensi Tunggal

Skenario *Inference* (Penggunaan Model):
1. Pengguna memasukkan satu gambar sampel (Contoh: Diupload lewat Flask App).
2. Sistem *Resize* dan konversi gambar ke *Grayscale*.
3. **Ekstraksi HOG** untuk mencari vektor 1D dari bentuk objek.
4. Nilai ini diskalakan kembali mengikuti **Scaler** *training*.
5. Dimasukkan ke persamaan Kernel SVM untuk dikalkulasikan (Keputusan Class \& Probabilitas Confidence).


In [ ]:
def predict_single_image(img_path, model, scaler):
    try:
        # Preprocessing & HOG
        img_rgb, img_gray = preprocess_image(img_path)
        features = extract_hog_features(img_gray)
        
        # Scaling
        features_scaled = scaler.transform([features])
        
        # Predict
        pred_label = model.predict(features_scaled)[0]
        prob = model.predict_proba(features_scaled)[0]
        
        # Mapping label
        label_text = 'Baik' if pred_label == 1 else 'Rusak'
        confidence = prob[pred_label] * 100
        
        # Visualisasi
        plt.figure(figsize=(4,4))
        plt.imshow(img_rgb)
        plt.title(f"Prediksi SVM: {label_text}\nConfidence: {confidence:.2f}%")
        plt.axis('off')
        plt.show()
    except Exception as e:
        print(f"Error: {e}")

if len(X) > 0 and len(image_paths) > 0:
    sample_test_image = image_paths[-1] # Ambil gambar contoh terakhir dari direktori
    predict_single_image(sample_test_image, best_model, scaler)
